# Laboratório Unity Catalog: Governança e Tabelas Delta

Até agora, salvamos os dados apenas como 'arquivos físicos' (.parquet). 
No Databricks moderno, usamos o **Unity Catalog** para registrar esses arquivos como **Tabelas Gerenciadas** (Managed Tables). 

Isso permite:
- Controle granular de acesso (GRANT/REVOKE)
- Consultas via SQL padrão por Analistas de Negócios
- Time Travel (voltar no tempo para versões antigas da tabela)
- Schema Evolution (evolução do formato dos dados sem quebrar o pipeline)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab: Unity Catalog") \
    .getOrCreate()

spark

### 1. Criando nosso Banco de Dados Lógico (Schema)

O Databricks usa a estrutura `catalog.schema.table`.
O catálogo padrão em contas gratuitas/novas chama-se `main` (ou `hive_metastore`).
Vamos criar um schema exclusivo para a nossa camada Silver.

In [2]:
# O bloco abaixo executa puro SQL no motor do Spark
spark.sql("CREATE SCHEMA IF NOT EXISTS main.e2e_silver")
print("Schema 'e2e_silver' garantido no catálogo 'main'!")

### 2. Lendo os arquivos Parquet (que otimizamos antes) e Salvando como Tabela Delta

In [3]:
ENVIRONMENT = "databricks_volume" # Altere para "local" quando rodar no Docker

if ENVIRONMENT == "local":
    BASE_PATH = "file:///home/jovyan/work/data"
elif ENVIRONMENT == "databricks_volume":
    # Novo caminho usando Unity Catalog Volumes (muito mais moderno que o antigo DBFS)
    BASE_PATH = "/Volumes/workspace/default/raw_data"

# Se vc fizer o upload dos CSVs soltos direto no volume, o raw_path é o próprio BASE_PATH
raw_path = f"{BASE_PATH}/"
silver_path = f"{BASE_PATH}/silver/churn_optimized.parquet"

print(f"[Config] Rodando no ambiente: {ENVIRONMENT}")
print(f"[Config] BASE_PATH = {BASE_PATH}\n")


### 3. Consultando a Tabela como um Analista de Dados

Agora que é uma tabela Delta, não precisamos mais saber onde o arquivo está guardado fisicamente.

In [4]:
df_query = spark.sql("SELECT * FROM main.e2e_silver.churn_optimized LIMIT 10")
display(df_query)

# PS: No Databricks Workspace, você também pode criar uma célula %sql e fazer:
# %sql
# SELECT * FROM main.e2e_silver.churn_optimized LIMIT 10